# Solar panel defect detector — RGB only

### What this model can and cannot do

Trained on visible-light imagery, so it detects what visible light shows:

**Can** — soiling and dust, bird droppings, cracked or shattered glass, delamination and
discoloration, vegetation shading, snow cover, missing or displaced modules.

**Cannot** — hotspots, cell and multi-cell defects, bypass-diode failure, offline modules.
These are *electrical* faults. They are visible in thermal infrared and essentially
invisible in RGB. No amount of training data fixes that; it is physics, not modelling. The
web app states this limitation to users, and so should you if you show this to anyone.

If you later add a thermal camera, the strongest public dataset is
[RaptorMaps InfraredSolarModules](https://github.com/RaptorMaps/InfraredSolarModules) —
20,000 real IR images, 12 classes, already cropped to single modules. Those crops suit a
classifier rather than a detector, which is a different (and much cheaper) pipeline.

### Getting data

You have no imagery of your own, so this trains on public sets. Roboflow Universe has
several RGB solar panel detection and soiling datasets; a free account gives you an API key
and export in YOLO format.

**Audit whatever you download before training on it.** The turbine dataset scored 0.782 and
was useless because of defects `tools/audit_dataset.py` catches in seconds. Assume a
scraped dataset has the same problems until the audit says otherwise.

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
import os, shutil, sys
from pathlib import Path

REPO = Path("/kaggle/input/drone-inspection")
WORK = Path("/kaggle/working/drone-inspection")
if not WORK.exists():
    shutil.copytree(REPO, WORK)
os.chdir(WORK); sys.path.insert(0, str(WORK))

# Option A - Roboflow. Put your key in Kaggle Secrets ("Add-ons > Secrets"), never inline.
# from kaggle_secrets import UserSecretsClient
# from roboflow import Roboflow
# key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
# dataset = Roboflow(api_key=key).workspace("WORKSPACE").project("PROJECT").version(1) \
#     .download("yolov11", location="/kaggle/working/solar_raw")

# Option B - attach a Kaggle Dataset you assembled yourself.
SOLAR = Path("/kaggle/working/solar_raw")
print("exists:", SOLAR.exists())

## Audit before training

Run this on the downloaded data. Two failures matter most:

* **source-family shortcut** — if the filenames reveal the class, or a class never shares an
  image with another, the model will learn the shortcut instead of the defect. This is
  exactly what ruined the turbine v1 model.
* **near-duplicate frame leakage** — solar farm imagery is usually captured as a flight
  sequence, so a random split puts adjacent frames on both sides. Validation then measures
  memorisation.

If either fails, re-split by capture group before training. `tools/rebuild_turbine.py` shows
the pattern; the contiguous-block split in `block_split()` is directly reusable.

In [ ]:
!python3 tools/audit_dataset.py {SOLAR}

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(SOLAR / "data.yaml"),
    epochs=60,
    imgsz=960,             # soiling patches and cell cracks are small
    batch=8,
    patience=12,

    optimizer="SGD",
    lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=5, cos_lr=True,

    cache="disk", device=0, workers=2, seed=0, deterministic=True,

    # Panels are rectilinear and photographed from many angles; rotation helps here more
    # than it does on blades. Keep hue jitter low - discoloration IS the signal for one of
    # the classes, so distorting colour trains the model to ignore what it should detect.
    degrees=15.0, fliplr=0.5, flipud=0.5, scale=0.4,
    hsv_h=0.005, hsv_s=0.5, hsv_v=0.3,
    mosaic=1.0, close_mosaic=10,

    save_period=10,
    project="/kaggle/working/runs", name="solar_v1", exist_ok=True,
    plots=True,
)

In [ ]:
!python3 tools/evaluate.py /kaggle/working/runs/solar_v1/weights/best.pt \
    --data {SOLAR}/data.yaml --split test --imgsz 960 \
    --out /kaggle/working/runs/eval_solar

## Export

`export_onnx.py` rewrites `web/models/manifest.json` with the class list from the trained
model, so the browser labels match the weights. Afterwards, **set the severity weights by
hand** — the exporter defaults every class to 1.0, and engineering judgement is what makes
that number mean something. A missing module is not the same event as some dust.

In [ ]:
!python3 tools/export_onnx.py /kaggle/working/runs/solar_v1/weights/best.pt \
    --name solar --imgsz 960
!cat web/models/manifest.json